Я хотел поизучать рекомендации основанные только на музыкальных фичах. Взял датасет Miillion songs(1%).
В основу взял timbre(MFCC)
И pitches(это вероятность => сила доминации классической ноты на промежутке времни, как я понял 0.5 секунды)

Основная идея была в том, что эти характеристики мы точно сможем достать из наших файлов mp3 и хотелось как то поработать именно сегментируя их. Также на этом датасете можно и сделать эмбеденги artist_terms, который представляет из себя списки с тегами артистов очень вариативные от rock до dirty south rap, я делал, но потом понял, что хочется попробовать построить расстояния только на аудиофичах причем не абстрактных, а технических

pitches и timbre - списки 12-ти мерных векторов, которые отвечают за MFCC{1-12} и каждую ноту для pitches. Независимо от длины списка я делил на 8 частей и для каждой выбирал статистики: mean, max, min, max - min

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

In [2]:
np.random.seed(42)

df = pd.read_csv("msd_8parts_features.csv")

titles = df[["artist_title", "artist_terms"]]
df.drop(["artist_title", "artist_terms"], axis=1, inplace=True)

print(df.shape)
print(titles.shape)

(10000, 842)
(10000, 2)


In [5]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

pca = PCA(n_components=150, random_state=42)
X_pca_150 = pca.fit_transform(df_scaled)

print(f"разложение объясняет {np.sum(pca.explained_variance_ratio_):.4f} дисперсии")

разложение объясняет 0.8416 дисперсии


Я делал анализ по разным количествам PCA 150 мне показалось самым оптимальным

In [13]:
def get_artist_name(artist_title: str) -> str:
    return artist_title.split(" - ")[0].strip()

Смотрим на совпадение по артисту или по совпадению тега артиста метрика не очень, потому что мало повторений артистов. И теги артиста достаточно странные и очень вариативные.

In [31]:
np.random.seed(42)
X_eval = X_pca_150

metrics = ["euclidean", "cosine", "manhattan", "chebyshev", "correlation", "canberra", "braycurtis"]
results = {}

for metric_name in metrics:
    knn = NearestNeighbors(n_neighbors=11, metric=metric_name, algorithm="brute")
    knn.fit(X_eval)

    sample_size = min(10000, len(X_eval))
    sample_indices = np.random.choice(len(X_eval), sample_size, replace=False)

    precisions = []

    for idx in sample_indices:
        distances, neighbors = knn.kneighbors([X_eval[idx]])
        neighbors = neighbors[0][1:]

        current_artist = get_artist_name(titles.iloc[idx]["artist_title"])
        current_tags = set(str(titles.iloc[idx]["artist_terms"]).split("|"))

        relevant = 0
        for n_idx in neighbors:
            neighbor_artist = get_artist_name(titles.iloc[n_idx]["artist_title"])
            neighbor_tags = set(str(titles.iloc[n_idx]["artist_terms"]).split("|"))

            if neighbor_artist == current_artist or len(current_tags.intersection(neighbor_tags)) > 0:
                relevant += 1

        precisions.append(relevant / 10)

    results[metric_name] = np.mean(precisions)

print("Метрика      Precision@10")
print("-" * 30)
for metric, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{metric:<12} {score:.4f}")

Метрика      Precision@10
------------------------------
braycurtis   0.5276
correlation  0.5255
cosine       0.5253
euclidean    0.5121
chebyshev    0.5048
manhattan    0.4832
canberra     0.4594


Лучшим оказалось Несходство Брея-Кертиса(braycurtis), что странно потому что используется зачастую в биоинформатике, далее в тестах использовал в основном косинусное расстояние из-за лучшей интепретации и практически неотличимого результата также и от correlation

In [51]:
example_idx = np.random.choice(len(X_pca_150), 1, replace=False)[0]

artist_title = titles.iloc[example_idx]["artist_title"]
artist_name = get_artist_name(artist_title)
track_name = artist_title.split(" - ")[1] if " - " in artist_title else artist_title

print(artist_title)
print({titles.iloc[example_idx]['artist_terms']})

knn = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute")
knn.fit(X_pca_150)

distances, neighbors = knn.kneighbors([X_pca_150[example_idx]])

print(f"Реки")
for i in range(1, 4):
    n_idx = neighbors[0][i]
    neighbor_title = titles.iloc[n_idx]["artist_title"]
    neighbor_artist = get_artist_name(neighbor_title)
    neighbor_track = neighbor_title.split(" - ")[1] if " - " in neighbor_title else neighbor_title

    print(f"  {neighbor_artist} - {neighbor_track}")

Mary Black - Flesh And Blood
{'piano|celtic|contemporary folk|new age|ballad|blues-rock|classical|downtempo|ambient|pop'}
Реки
  Restless Heart - That Rock Won't Roll
  Pat Green - Threadbare Gypsy Soul
  Sophie B. Hawkins - 32 Lines (Album Version)


Переслушал кучу треков работает странно. Часто похоже по идее звучания например к какой-то акустике(гитаре) медленной мне часто попадались треки с медленным пианино. Как мне показалось хорошо находит похожие треки, на женский вокал. Также хорошо работает если в треке часто протягиваются слова. Иногда просто вообще не похоже. Точность на метриках в принципе оправдывает. Оставил в выводе пример как мне кажется хороших реков